# TekaRx full-GNN ablation study

This Colab notebook measures which parts of the trained TekaRx model contribute useful validation signal. It restores the **exact versioned `gnn-full` graph and model checkpoints** from Google Drive, retrains controlled GNN ablations, and compares them with the saved XGBoost baseline.

> TekaRx outputs are research decision-support signals, not diagnoses or causal estimates.

Use a GPU runtime. The notebook evaluates **2024Q1 validation only**. It never reads 2024Q2 test labels, never selects a threshold on test, and never passes `--evaluate-test`. Expect roughly 45–75 minutes for the three retraining arms on a T4, plus restore and scoring time. All restartable training checkpoints and final study artifacts are stored under `MyDrive/Teka-Rx-full/data/processed/ablation_studies/`.

## Experimental design

| Arm | What changes | Interpretation |
|---|---|---|
| Full GNN | Reuse the verified selected model | Reference |
| No normalized dosage | Retrain after removing normalized dosage columns | Incremental dosage value |
| Patient only | Retrain with every drug feature set to zero | Value beyond patient tabular features |
| Shuffled topology | Retrain after rotating drug endpoints independently inside train/validation/test | Value of the correct regimen-to-drug assignment |
| Saved XGBoost | Score the matching graph checkpoint's baseline | Strong non-neural comparator |

The current GNN is a one-hop patient encoder plus the mean of fixed drug features. Therefore, `shuffled_topology` tests regimen alignment; it is not a test of multi-hop patient-to-drug-to-patient message passing. Rotation preserves each patient's edge count and each split's drug-frequency multiset, but it can create duplicate patient–drug pairs.

Threshold-dependent metrics are reported at: (1) a configurable fixed threshold of 0.50, (2) the validation threshold maximizing F1, and (3) the highest validation threshold attaining at least 95% recall. The 0.50 result is a diagnostic, not a clinical default: class-weighted GNN training does not guarantee calibrated probabilities. Policies 2 and 3 are exploratory because selection and reporting use the same validation period. Freeze one policy before the one-time final test evaluation.

In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("This notebook must run in a Google Colab runtime.") from exc

drive.mount("/content/drive")


In [ ]:
from __future__ import annotations

import copy
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import shlex
import shutil
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

REPO_URL = "https://github.com/matthew-sudo2/Teka-Rx.git"
REPO_DIR = Path("/content/Teka-Rx")
DRIVE_DATA = Path("/content/drive/MyDrive/Teka-Rx-full/data")
LOCAL_DATA = Path("/content/tekarx-ablation-data")

DECISION_THRESHOLD = 0.50
FIXED_THRESHOLD_POLICY = f"fixed_{DECISION_THRESHOLD:.2f}"
TARGET_RECALL = 0.95
THRESHOLD_SWEEP = np.round(np.arange(0.05, 0.951, 0.01), 2)
SEED = 42
EPOCHS = 100
PATIENCE = 15
BATCH_SIZE = 8192
HIDDEN_CHANNELS = 64
DROPOUT = 0.20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EDGE_CHUNK_SIZE = 250_000
PREDICTION_BATCH_SIZE = 8192
SCAN_CHUNK_SIZE = 250_000
VARIANT_FORMAT_VERSION = 1

if not 0 < DECISION_THRESHOLD < 1:
    raise ValueError("DECISION_THRESHOLD must be between zero and one.")
if not 0 < TARGET_RECALL <= 1:
    raise ValueError("TARGET_RECALL must be in (0, 1].")
if not DRIVE_DATA.is_dir():
    raise FileNotFoundError(f"Missing Drive data root: {DRIVE_DATA}")
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
print(f"Durable checkpoint root: {DRIVE_DATA}")
print(f"Fast local scratch: {LOCAL_DATA}")


In [ ]:
def gibibytes(value: int) -> float:
    return value / (1024**3)


def run_command(*parts: object, cwd: Path | None = None) -> subprocess.CompletedProcess[str]:
    command = [str(part) for part in parts]
    print("$", shlex.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True, text=True)


def load_json(path: Path) -> dict[str, object]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json_atomic(path: Path, payload: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(16 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def tree_inventory(root: Path) -> dict[str, int]:
    return {
        path.relative_to(root).as_posix(): path.stat().st_size
        for path in root.rglob("*")
        if path.is_file()
    }


def copy_with_progress(source: Path, destination: Path, *, label: str) -> None:
    if not source.is_file():
        raise FileNotFoundError(f"Missing {label}: {source}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and destination.stat().st_size == source.stat().st_size:
        return
    temporary = destination.with_suffix(destination.suffix + ".copying")
    with source.open("rb") as incoming, temporary.open("wb") as outgoing, tqdm(
        total=source.stat().st_size, desc=label, unit="B", unit_scale=True, dynamic_ncols=True
    ) as progress:
        while True:
            chunk = incoming.read(16 * 1024 * 1024)
            if not chunk:
                break
            outgoing.write(chunk)
            progress.update(len(chunk))
        outgoing.flush()
        os.fsync(outgoing.fileno())
    shutil.copystat(source, temporary)
    if temporary.stat().st_size != source.stat().st_size:
        raise RuntimeError(f"Incomplete copy: {label}")
    os.replace(temporary, destination)


def validate_checkpoint(root: Path, artifacts: dict[str, object], *, label: str) -> None:
    if not root.is_dir() or not artifacts:
        raise RuntimeError(f"Missing or empty {label} checkpoint: {root}")
    inventory = tree_inventory(root)
    if set(inventory) != set(artifacts):
        missing = sorted(set(artifacts) - set(inventory))[:5]
        extra = sorted(set(inventory) - set(artifacts))[:5]
        raise RuntimeError(f"{label} inventory mismatch; missing={missing}, extra={extra}")
    for relative, metadata in artifacts.items():
        if not isinstance(metadata, dict):
            raise RuntimeError(f"Invalid {label} metadata: {relative}")
        path = root / relative
        if path.stat().st_size != metadata.get("size_bytes"):
            raise RuntimeError(f"{label} size mismatch: {relative}")
        expected_hash = metadata.get("sha256")
        if expected_hash is not None and sha256_file(path) != expected_hash:
            raise RuntimeError(f"{label} SHA-256 mismatch: {relative}")


def restore_files(
    source_root: Path, destination_root: Path, relatives: set[str],
    artifacts: dict[str, object], *, label: str,
) -> None:
    required_bytes = sum(
        int(artifacts[relative]["size_bytes"])
        for relative in relatives
        if not (destination_root / relative).is_file()
        or (destination_root / relative).stat().st_size != artifacts[relative]["size_bytes"]
    )
    free_bytes = shutil.disk_usage("/content").free
    print(
        f"{label}: {gibibytes(required_bytes):.2f} GiB pending; "
        f"{gibibytes(free_bytes):.2f} GiB local free", flush=True,
    )
    if free_bytes < required_bytes + 8 * 1024**3:
        raise RuntimeError(f"Insufficient /content headroom to restore {label}.")
    for relative in sorted(relatives):
        copy_with_progress(
            source_root / relative, destination_root / relative,
            label=f"{label}: {relative}",
        )
        metadata = artifacts[relative]
        restored = destination_root / relative
        if restored.stat().st_size != metadata["size_bytes"]:
            raise RuntimeError(f"Restored {label} size mismatch: {relative}")
        if metadata.get("sha256") and sha256_file(restored) != metadata["sha256"]:
            raise RuntimeError(f"Restored {label} hash mismatch: {relative}")


In [ ]:
DRIVE_PROCESSED = DRIVE_DATA / "processed"
graph_marker_path = DRIVE_PROCESSED / "_GRAPH_SUCCESS.json"
gnn_marker_path = DRIVE_PROCESSED / "_GNN_SUCCESS.json"
if not graph_marker_path.is_file() or not gnn_marker_path.is_file():
    raise FileNotFoundError("The verified graph and GNN success markers are required.")

graph_marker = load_json(graph_marker_path)
gnn_marker = load_json(gnn_marker_path)
if graph_marker.get("checkpoint") != "verified_complete":
    raise RuntimeError("Graph checkpoint is not marked verified_complete.")
if gnn_marker.get("checkpoint") != "verified_complete":
    raise RuntimeError("GNN checkpoint is not marked verified_complete.")
if graph_marker.get("split_preset") != "gnn-full" or gnn_marker.get("split_preset") != "gnn-full":
    raise RuntimeError("This study requires the gnn-full temporal split.")
if gnn_marker.get("test_evaluated") is not False:
    raise RuntimeError("The selected GNN marker does not preserve the locked test.")
graph_checkpoint_id = graph_marker.get("graph_checkpoint_id")
if not isinstance(graph_checkpoint_id, str) or gnn_marker.get("graph_checkpoint_id") != graph_checkpoint_id:
    raise RuntimeError("GNN and graph checkpoints do not match.")
gnn_checkpoint_id = gnn_marker.get("gnn_checkpoint_id")
if not isinstance(gnn_checkpoint_id, str):
    raise RuntimeError("GNN marker has no versioned checkpoint ID.")
CHECKPOINT_GIT_SHA = graph_marker.get("git_sha")
if not isinstance(CHECKPOINT_GIT_SHA, str) or len(CHECKPOINT_GIT_SHA) != 40:
    raise RuntimeError("Graph marker must contain an immutable 40-character Git SHA.")
if gnn_marker.get("git_sha") != CHECKPOINT_GIT_SHA:
    raise RuntimeError("GNN and graph were produced by different code revisions.")
STUDY_ID = f"{graph_checkpoint_id}-ablation-v{VARIANT_FORMAT_VERSION}-seed{SEED}"
DRIVE_STUDY = DRIVE_PROCESSED / "ablation_studies" / STUDY_ID
LOCAL_STUDY = LOCAL_DATA / "processed" / "ablation_studies" / STUDY_ID
DRIVE_STUDY.mkdir(parents=True, exist_ok=True)
LOCAL_STUDY.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    "graph_checkpoint_id": graph_checkpoint_id,
    "gnn_checkpoint_id": gnn_checkpoint_id,
    "checkpoint_git_sha": CHECKPOINT_GIT_SHA,
    "study_id": STUDY_ID,
    "test_evaluated": False,
}, indent=2))


In [ ]:
if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
    raise RuntimeError(f"Remove or rename the incomplete repository directory: {REPO_DIR}")
if not (REPO_DIR / ".git").is_dir():
    run_command("git", "clone", REPO_URL, REPO_DIR)
run_command("git", "-C", REPO_DIR, "fetch", "--prune", "origin")
run_command("git", "-C", REPO_DIR, "checkout", "--detach", CHECKPOINT_GIT_SHA)
RESOLVED_GIT_SHA = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
if RESOLVED_GIT_SHA != CHECKPOINT_GIT_SHA:
    raise RuntimeError("Failed to check out the exact graph/model code revision.")
run_command(
    sys.executable, "-m", "pip", "install", "-e",
    f"{REPO_DIR}[graph,notebook]", "scikit-learn>=1.5,<2", "matplotlib>=3.9,<4",
    "tabulate>=0.9,<1",
)
repo_source = str((REPO_DIR / "src").resolve())
if repo_source not in sys.path:
    sys.path.insert(0, repo_source)
importlib.invalidate_caches()
import tekarx
print(f"TekaRx loaded from: {Path(tekarx.__file__).resolve()}")
print(f"Exact training revision: {RESOLVED_GIT_SHA}")


In [ ]:
import matplotlib
import pandas as pd
import pyarrow
import sklearn
import torch
import xgboost as xgb

versions = {
    "python": platform.python_version(),
    "tekarx": importlib.metadata.version("tekarx"),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pyarrow": pyarrow.__version__,
    "torch": torch.__version__,
    "xgboost": xgb.__version__,
    "scikit_learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
}
print(json.dumps(versions, indent=2))
print(f"Local /content free: {gibibytes(shutil.disk_usage('/content').free):.2f} GiB")
print(f"Drive free: {gibibytes(shutil.disk_usage(DRIVE_DATA).free):.2f} GiB")
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU, then rerun.")
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB)")
print(f"Torch CUDA runtime: {torch.version.cuda}")


In [ ]:
graph_artifacts = graph_marker.get("artifacts")
gnn_artifacts = gnn_marker.get("artifacts")
if not isinstance(graph_artifacts, dict) or not isinstance(gnn_artifacts, dict):
    raise RuntimeError("Checkpoint markers lack artifact inventories.")
drive_graph_root = (DRIVE_PROCESSED / str(graph_marker["checkpoint_dir"])).resolve()
drive_gnn_root = (DRIVE_PROCESSED / str(gnn_marker["checkpoint_dir"])).resolve()
if drive_graph_root.parent != (DRIVE_PROCESSED / "graph_checkpoints").resolve():
    raise RuntimeError("Unsafe graph checkpoint path.")
if drive_gnn_root.parent != (DRIVE_PROCESSED / "gnn_checkpoints").resolve():
    raise RuntimeError("Unsafe GNN checkpoint path.")
validate_checkpoint(drive_graph_root, graph_artifacts, label="Drive graph")
validate_checkpoint(drive_gnn_root, gnn_artifacts, label="Drive GNN")

graph_relatives = {
    relative for relative in graph_artifacts
    if relative == "tekarx_graph.pt"
    or relative == "graph_manifest.json"
    or relative == "xgboost_baseline.json"
    or relative.startswith("tekarx_graph_arrays/")
}
for required in ("tekarx_graph.pt", "graph_manifest.json", "xgboost_baseline.json", "tekarx_graph_arrays/manifest.json"):
    if required not in graph_relatives:
        raise RuntimeError(f"Graph checkpoint lacks {required}")
gnn_relatives = {
    "tekarx_inductive_gnn.pt",
    "tekarx_inductive_gnn_manifest.json",
    "colab_training_metadata.json",
}
if not gnn_relatives.issubset(gnn_artifacts):
    raise RuntimeError(f"GNN checkpoint lacks {sorted(gnn_relatives - set(gnn_artifacts))}")

local_graph_root = LOCAL_DATA / "processed" / "graph_checkpoints" / graph_checkpoint_id
local_gnn_root = LOCAL_DATA / "processed" / "gnn_checkpoints" / gnn_checkpoint_id
restore_files(
    drive_graph_root, local_graph_root, graph_relatives, graph_artifacts, label="full graph"
)
restore_files(
    drive_gnn_root, local_gnn_root, gnn_relatives, gnn_artifacts, label="selected GNN"
)
RESTORED_GRAPH_PATH = local_graph_root / "tekarx_graph.pt"
FULL_GNN_MODEL_PATH = local_gnn_root / "tekarx_inductive_gnn.pt"
XGB_MODEL_PATH = local_graph_root / "xgboost_baseline.json"
full_payload = torch.load(FULL_GNN_MODEL_PATH, map_location="cpu", weights_only=True)
if not isinstance(full_payload, dict) or full_payload.get("test_auc") is not None:
    raise RuntimeError("Selected GNN model is invalid or has consumed the locked test.")
print("Verified graph, full GNN, and matching XGBoost checkpoints restored.")
print(f"Recorded full-GNN validation AUROC: {float(full_payload['validation_auc']):.6f}")
del full_payload


In [ ]:
from tekarx.transform.graph_storage import load_graph_arrays

base_bundle = load_graph_arrays(RESTORED_GRAPH_PATH, mmap_mode="r")
required_arrays = {
    "patient_x", "patient_y", "patient_split_id", "patient_primaryid",
    "drug_x", "edge_patient_index", "edge_drug_index",
}
if not required_arrays.issubset(base_bundle.arrays):
    raise RuntimeError(f"Graph arrays missing: {sorted(required_arrays - set(base_bundle.arrays))}")
split_counts = {0: 0, 1: 0, 2: 0}
split_values = base_bundle.arrays["patient_split_id"]
for start in tqdm(range(0, split_values.shape[0], SCAN_CHUNK_SIZE), desc="Auditing patient splits"):
    local_split = np.asarray(split_values[start : start + SCAN_CHUNK_SIZE])
    if not np.isin(local_split, (0, 1, 2)).all():
        raise RuntimeError("Unexpected split ID in graph.")
    for split_value in split_counts:
        split_counts[split_value] += int(np.count_nonzero(local_split == split_value))
if any(count == 0 for count in split_counts.values()):
    raise RuntimeError(f"A required temporal split is empty: {split_counts}")
graph_audit = pd.DataFrame([
    {"split": name, "split_id": value, "patients": split_counts[value]}
    for value, name in ((0, "train"), (1, "validation"), (2, "test_locked"))
])
display(graph_audit)
print(f"Patient feature columns: {base_bundle.arrays['patient_x'].shape[1]}")
print(f"Drug nodes/features: {base_bundle.arrays['drug_x'].shape}")
print(f"Exposure edges: {base_bundle.arrays['edge_patient_index'].shape[0]:,}")
print("Leakage guard: no label value for split_id == 2 was indexed.")
del base_bundle, split_values
gc.collect()


In [ ]:
def hardlink_or_copy(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.link(source, destination)
    except OSError:
        copy_with_progress(source, destination, label=f"variant: {source.name}")


def copy_rotated_range(
    source: np.ndarray, destination: np.memmap,
    source_start: int, source_stop: int, destination_start: int, progress: tqdm,
) -> None:
    cursor = source_start
    output = destination_start
    while cursor < source_stop:
        stop = min(cursor + EDGE_CHUNK_SIZE, source_stop)
        values = np.asarray(source[cursor:stop])
        destination[output : output + values.shape[0]] = values
        output += values.shape[0]
        cursor = stop
        progress.update(values.shape[0])


def build_graph_variant(variant: str) -> Path:
    if variant not in {"patient_only", "shuffled_topology"}:
        raise ValueError(f"Unsupported graph variant: {variant}")
    source_bundle = load_graph_arrays(RESTORED_GRAPH_PATH, mmap_mode="r")
    target_root = LOCAL_STUDY / "graphs" / variant
    success_path = target_root / "_SUCCESS.json"
    if success_path.is_file():
        success = load_json(success_path)
        descriptor = target_root / "tekarx_graph.pt"
        if (
            success.get("base_graph_checkpoint_id") == graph_checkpoint_id
            and success.get("variant") == variant
            and success.get("variant_format_version") == VARIANT_FORMAT_VERSION
            and descriptor.is_file()
        ):
            load_graph_arrays(descriptor, mmap_mode="r")
            print(f"Reusing local {variant} graph variant.")
            return descriptor
    if target_root.exists():
        resolved = target_root.resolve()
        expected_parent = (LOCAL_STUDY / "graphs").resolve()
        if resolved.parent != expected_parent:
            raise RuntimeError(f"Unsafe variant cleanup target: {resolved}")
        shutil.rmtree(resolved)
    arrays_root = target_root / "tekarx_graph_arrays"
    arrays_root.mkdir(parents=True)
    source_arrays_root = source_bundle.manifest_path.parent
    manifest = copy.deepcopy(source_bundle.manifest)
    special_name = "drug_x" if variant == "patient_only" else "edge_drug_index"
    for name, metadata in manifest["arrays"].items():
        relative = Path(str(metadata["path"]))
        source = source_arrays_root / relative
        destination = arrays_root / relative
        if name != special_name:
            hardlink_or_copy(source, destination)
            continue
        original = source_bundle.arrays[name]
        destination.parent.mkdir(parents=True, exist_ok=True)
        changed = np.lib.format.open_memmap(
            destination, mode="w+", dtype=original.dtype, shape=original.shape
        )
        if variant == "patient_only":
            for start in tqdm(range(0, original.shape[0], SCAN_CHUNK_SIZE), desc="Zeroing drug features"):
                changed[start : start + SCAN_CHUNK_SIZE] = 0
        else:
            split_offsets = manifest.get("edge_order", {}).get("split_offsets", {})
            ordered = [split_offsets.get(name) for name in ("train", "validation", "test")]
            if any(not isinstance(bounds, list) or len(bounds) != 2 for bounds in ordered):
                raise RuntimeError("Graph lacks complete edge split offsets.")
            if ordered[0][0] != 0 or ordered[-1][1] != original.shape[0]:
                raise RuntimeError("Edge split offsets do not cover the edge array.")
            with tqdm(total=original.shape[0], desc="Rotating drug endpoints", unit="edge") as progress:
                for split_index, (start, stop) in enumerate(ordered):
                    count = int(stop) - int(start)
                    if count < 2:
                        raise RuntimeError("Every split needs at least two edges for shuffling.")
                    shift = 1 + ((SEED * 1_000_003 + split_index * 97) % (count - 1))
                    first_count = count - shift
                    copy_rotated_range(original, changed, int(start) + shift, int(stop), int(start), progress)
                    copy_rotated_range(original, changed, int(start), int(start) + shift, int(start) + first_count, progress)
            manifest["edge_order"]["sort"] = ["patient_split_id", "patient_index"]
        changed.flush()
        del changed
    metadata_relative = Path(str(manifest["drug_metadata_path"]))
    hardlink_or_copy(source_arrays_root / metadata_relative, arrays_root / metadata_relative)
    manifest["ablation"] = {
        "variant": variant,
        "variant_format_version": VARIANT_FORMAT_VERSION,
        "base_graph_checkpoint_id": graph_checkpoint_id,
        "seed": SEED,
        "split_local_rotation": variant == "shuffled_topology",
        "patient_degree_preserved": True,
        "split_drug_frequency_preserved": variant == "shuffled_topology",
        "test_labels_accessed": False,
    }
    write_json_atomic(arrays_root / "manifest.json", manifest)
    descriptor = target_root / "tekarx_graph.pt"
    temporary_descriptor = descriptor.with_suffix(".pt.tmp")
    torch.save({
        "format": "tekarx.memmap_graph",
        "format_version": 1,
        "manifest_path": "tekarx_graph_arrays/manifest.json",
    }, temporary_descriptor)
    os.replace(temporary_descriptor, descriptor)
    load_graph_arrays(descriptor, mmap_mode="r")
    write_json_atomic(success_path, {
        "variant": variant,
        "variant_format_version": VARIANT_FORMAT_VERSION,
        "base_graph_checkpoint_id": graph_checkpoint_id,
        "created_at_utc": datetime.now(UTC).isoformat(),
        "test_evaluated": False,
    })
    del source_bundle
    gc.collect()
    return descriptor


PATIENT_ONLY_GRAPH_PATH = build_graph_variant("patient_only")
SHUFFLED_GRAPH_PATH = build_graph_variant("shuffled_topology")
print("Ablation graph variants are ready. The source graph was not modified.")


In [ ]:
from tekarx.modeling.gnn import train_inductive_gnn

ARM_SPECS = {
    "no_normalized_dosage": (RESTORED_GRAPH_PATH, "prospective-no-dosage"),
    "patient_only": (PATIENT_ONLY_GRAPH_PATH, "prospective"),
    "shuffled_topology": (SHUFFLED_GRAPH_PATH, "prospective"),
}
MODEL_ROOT = LOCAL_STUDY / "models"
DRIVE_MODEL_ROOT = DRIVE_STUDY / "models"
CHECKPOINT_ROOT = DRIVE_STUDY / "training_checkpoints"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_MODEL_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)


def validate_locked_model(path: Path) -> dict[str, object]:
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict) or payload.get("test_auc") is not None:
        raise RuntimeError(f"Model is invalid or has evaluated test: {path}")
    return payload


def train_or_restore_arm(arm: str, graph_path: Path, feature_track: str) -> Path:
    drive_arm = DRIVE_MODEL_ROOT / arm
    drive_model = drive_arm / "model.pt"
    drive_manifest = drive_arm / "training_manifest.json"
    drive_model_manifest = drive_arm / "model_manifest.json"
    local_arm = MODEL_ROOT / arm
    local_model = local_arm / "model.pt"
    if drive_model.is_file() and drive_manifest.is_file() and drive_model_manifest.is_file():
        metadata = load_json(drive_manifest)
        if (
            metadata.get("arm") == arm
            and metadata.get("graph_checkpoint_id") == graph_checkpoint_id
            and metadata.get("git_sha") == RESOLVED_GIT_SHA
            and metadata.get("test_evaluated") is False
            and metadata.get("model_sha256") == sha256_file(drive_model)
            and metadata.get("model_manifest_sha256") == sha256_file(drive_model_manifest)
        ):
            copy_with_progress(drive_model, local_model, label=f"restore {arm} model")
            if sha256_file(local_model) != metadata["model_sha256"]:
                raise RuntimeError(f"Restored {arm} model hash mismatch.")
            validate_locked_model(local_model)
            print(f"Reusing completed arm: {arm}")
            return local_model
    checkpoint = CHECKPOINT_ROOT / f"{arm}.pt"
    print(f"\nTraining arm: {arm}", flush=True)
    record = train_inductive_gnn(
        data_dir=LOCAL_DATA,
        graph_path=graph_path,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        hidden_channels=HIDDEN_CHANNELS,
        dropout=DROPOUT,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        patience=PATIENCE,
        device="cuda",
        seed=SEED,
        edge_chunk_size=EDGE_CHUNK_SIZE,
        evaluate_test=False,
        feature_track=feature_track,
        checkpoint_path=checkpoint,
        resume_from=checkpoint if checkpoint.is_file() else None,
        checkpoint_every=5,
    )
    source_model = Path(record.model_path)
    source_model_manifest = Path(record.manifest_path)
    payload = validate_locked_model(source_model)
    local_arm.mkdir(parents=True, exist_ok=True)
    local_model_manifest = local_arm / "model_manifest.json"
    copy_with_progress(source_model, local_model, label=f"stage {arm} model")
    copy_with_progress(
        source_model_manifest, local_model_manifest, label=f"stage {arm} model manifest"
    )
    metadata = {
        "arm": arm,
        "feature_track": feature_track,
        "graph_checkpoint_id": graph_checkpoint_id,
        "base_gnn_checkpoint_id": gnn_checkpoint_id,
        "git_sha": RESOLVED_GIT_SHA,
        "completed_at_utc": datetime.now(UTC).isoformat(),
        "validation_auc": float(payload["validation_auc"]),
        "best_epoch": int(payload["best_epoch"]),
        "model_sha256": sha256_file(local_model),
        "model_manifest_sha256": sha256_file(local_model_manifest),
        "test_evaluated": False,
        "hyperparameters": {
            "epochs": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE,
            "hidden_channels": HIDDEN_CHANNELS, "dropout": DROPOUT,
            "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
            "seed": SEED, "edge_chunk_size": EDGE_CHUNK_SIZE,
        },
    }
    drive_arm.mkdir(parents=True, exist_ok=True)
    copy_with_progress(local_model, drive_model, label=f"publish {arm} model")
    copy_with_progress(
        local_model_manifest, drive_model_manifest, label=f"publish {arm} model manifest"
    )
    if sha256_file(drive_model) != metadata["model_sha256"]:
        raise RuntimeError(f"Published {arm} model hash mismatch.")
    if sha256_file(drive_model_manifest) != metadata["model_manifest_sha256"]:
        raise RuntimeError(f"Published {arm} model-manifest hash mismatch.")
    write_json_atomic(drive_manifest, metadata)
    return local_model


arm_models = {"full_gnn": FULL_GNN_MODEL_PATH}
for arm, (graph_path, feature_track) in ARM_SPECS.items():
    arm_models[arm] = train_or_restore_arm(arm, graph_path, feature_track)
display(pd.DataFrame([
    {"arm": arm, "model": str(path), "test_evaluated": False}
    for arm, path in arm_models.items()
]))


In [ ]:
from torch import nn
from tekarx.modeling.gnn import (
    _aggregate_drug_neighbors_memmap,
    _close_memmap,
    _load_training_graph,
    _make_model,
    _predict_split,
)

device = torch.device("cuda")
base_bundle = load_graph_arrays(RESTORED_GRAPH_PATH, mmap_mode="r")
patient_count = int(base_bundle.arrays["patient_split_id"].shape[0])
validation_count = split_counts[1]
validation_primaryid = np.empty(validation_count, dtype=np.int64)
validation_truth = np.empty(validation_count, dtype=np.int8)
offset = 0
for start in tqdm(range(0, patient_count, SCAN_CHUNK_SIZE), desc="Collecting validation identity"):
    stop = min(start + SCAN_CHUNK_SIZE, patient_count)
    local_split = np.asarray(base_bundle.arrays["patient_split_id"][start:stop])
    validation_local = np.flatnonzero(local_split == 1)
    count = validation_local.shape[0]
    if count:
        global_ids = validation_local + start
        validation_primaryid[offset : offset + count] = np.asarray(
            base_bundle.arrays["patient_primaryid"][global_ids], dtype=np.int64
        )
        validation_truth[offset : offset + count] = np.asarray(
            base_bundle.arrays["patient_y"][global_ids], dtype=np.int8
        )
        offset += count
if offset != validation_count or not np.isin(validation_truth, (0, 1)).all():
    raise RuntimeError("Validation identity/label collection failed.")
if np.unique(validation_primaryid).shape[0] != validation_count:
    raise RuntimeError("Validation primaryid values are not unique.")
del base_bundle
gc.collect()


def create_neighbor_cache(graph_path: Path, cache_name: str, *, zero: bool = False):
    training_graph = _load_training_graph(
        graph_path, validation_chunk_size=SCAN_CHUNK_SIZE, torch=torch
    )
    cache_path = LOCAL_STUDY / f".{cache_name}-neighbors.npy"
    cache_path.unlink(missing_ok=True)
    if zero:
        neighbors = np.lib.format.open_memmap(
            cache_path, mode="w+", dtype=np.float32,
            shape=(training_graph.patient_count, training_graph.drug_x.shape[1]),
        )
        for start in tqdm(range(0, training_graph.patient_count, SCAN_CHUNK_SIZE), desc="Creating zero neighbors"):
            neighbors[start : start + SCAN_CHUNK_SIZE] = 0
        neighbors.flush()
    else:
        neighbors = _aggregate_drug_neighbors_memmap(
            patient_count=training_graph.patient_count,
            drug_features=training_graph.drug_x,
            edge_patient_index=training_graph.edge_patient_index,
            edge_drug_index=training_graph.edge_drug_index,
            output_path=cache_path,
            edge_chunk_size=EDGE_CHUNK_SIZE,
            torch=torch,
        )
    return training_graph, neighbors, cache_path


def score_gnn_model(model_path: Path, training_graph, neighbors) -> np.ndarray:
    payload = validate_locked_model(model_path)
    architecture = payload["architecture"]
    feature_names = tuple(payload["patient_feature_names"])
    feature_index = {name: index for index, name in enumerate(training_graph.feature_names)}
    if any(name not in feature_index for name in feature_names):
        raise RuntimeError(f"Model features do not match graph: {model_path}")
    selected_indices = [feature_index[name] for name in feature_names]
    model = _make_model(
        nn,
        patient_channels=int(architecture["patient_channels"]),
        drug_channels=int(architecture["drug_channels"]),
        hidden_channels=int(architecture["hidden_channels"]),
        dropout=float(architecture["dropout"]),
    )
    model.load_state_dict(payload["model_state_dict"])
    model.to(device)
    normalization = payload["normalization"]
    scores, truth = _predict_split(
        model,
        patient_x=training_graph.patient_x,
        drug_neighbors=neighbors,
        labels=training_graph.labels,
        split_id=training_graph.split_id,
        split_value=1,
        patient_feature_indices=selected_indices,
        patient_mean=normalization["patient_mean"],
        patient_std=normalization["patient_std"],
        neighbor_mean=normalization["drug_neighbor_mean"],
        neighbor_std=normalization["drug_neighbor_std"],
        row_count=training_graph.patient_count,
        scan_chunk_size=SCAN_CHUNK_SIZE,
        batch_size=PREDICTION_BATCH_SIZE,
        device=device,
        torch=torch,
    )
    if not np.array_equal(truth, validation_truth):
        raise RuntimeError(f"Validation labels are misaligned for {model_path}")
    del model, payload, truth
    torch.cuda.empty_cache()
    return scores


predictions: dict[str, np.ndarray] = {}
base_graph, base_neighbors, base_cache = create_neighbor_cache(RESTORED_GRAPH_PATH, "base")
try:
    predictions["full_gnn"] = score_gnn_model(arm_models["full_gnn"], base_graph, base_neighbors)
    predictions["no_normalized_dosage"] = score_gnn_model(arm_models["no_normalized_dosage"], base_graph, base_neighbors)
finally:
    _close_memmap(base_neighbors)
    del base_neighbors, base_graph
    gc.collect()
    base_cache.unlink(missing_ok=True)

patient_graph, patient_neighbors, patient_cache = create_neighbor_cache(
    PATIENT_ONLY_GRAPH_PATH, "patient-only", zero=True
)
try:
    predictions["patient_only"] = score_gnn_model(arm_models["patient_only"], patient_graph, patient_neighbors)
finally:
    _close_memmap(patient_neighbors)
    del patient_neighbors, patient_graph
    gc.collect()
    patient_cache.unlink(missing_ok=True)

shuffled_graph, shuffled_neighbors, shuffled_cache = create_neighbor_cache(
    SHUFFLED_GRAPH_PATH, "shuffled"
)
try:
    predictions["shuffled_topology"] = score_gnn_model(
        arm_models["shuffled_topology"], shuffled_graph, shuffled_neighbors
    )
finally:
    _close_memmap(shuffled_neighbors)
    del shuffled_neighbors, shuffled_graph
    gc.collect()
    shuffled_cache.unlink(missing_ok=True)

base_bundle = load_graph_arrays(RESTORED_GRAPH_PATH, mmap_mode="r")
booster = xgb.Booster()
booster.load_model(XGB_MODEL_PATH)
if booster.num_features() != base_bundle.arrays["patient_x"].shape[1]:
    raise RuntimeError("Saved XGBoost feature count does not match the graph.")
xgb_scores = np.empty(validation_count, dtype=np.float32)
offset = 0
for start in tqdm(range(0, patient_count, SCAN_CHUNK_SIZE), desc="Scoring saved XGBoost"):
    stop = min(start + SCAN_CHUNK_SIZE, patient_count)
    local_split = np.asarray(base_bundle.arrays["patient_split_id"][start:stop])
    mask = local_split == 1
    count = int(np.count_nonzero(mask))
    if count:
        features = np.ascontiguousarray(
            np.asarray(base_bundle.arrays["patient_x"][start:stop])[mask], dtype=np.float32
        )
        xgb_scores[offset : offset + count] = booster.inplace_predict(features)
        offset += count
if offset != validation_count:
    raise RuntimeError("XGBoost validation prediction count is inconsistent.")
predictions["xgboost_saved_baseline"] = xgb_scores
del booster, base_bundle, xgb_scores
gc.collect()
print("Validation-only predictions complete. The test labels remain locked.")


In [ ]:
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
)


def safe_ratio(numerator: int, denominator: int) -> float:
    return float(numerator / denominator) if denominator else 0.0


def max_f1_threshold(y_true: np.ndarray, scores: np.ndarray) -> float:
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    if thresholds.size == 0:
        return DECISION_THRESHOLD
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best = np.flatnonzero(np.isclose(f1, np.nanmax(f1), rtol=0, atol=1e-12))
    return float(thresholds[best[-1]])


def recall_target_threshold(y_true: np.ndarray, scores: np.ndarray, target: float) -> float:
    _, recall, thresholds = precision_recall_curve(y_true, scores)
    feasible = np.flatnonzero(recall[:-1] >= target)
    if feasible.size == 0:
        return 0.0
    return float(thresholds[feasible[-1]])


def expected_calibration_error(y_true: np.ndarray, scores: np.ndarray, bins: int = 15) -> float:
    edges = np.linspace(0, 1, bins + 1)
    total = y_true.shape[0]
    error = 0.0
    for index in range(bins):
        upper_inclusive = index == bins - 1
        mask = (scores >= edges[index]) & (scores <= edges[index + 1] if upper_inclusive else scores < edges[index + 1])
        if np.any(mask):
            error += float(np.count_nonzero(mask) / total) * abs(float(scores[mask].mean()) - float(y_true[mask].mean()))
    return error


def classification_metrics(
    model: str, y_true: np.ndarray, scores: np.ndarray,
    threshold: float, threshold_policy: str, global_metrics: dict[str, float],
) -> dict[str, object]:
    predicted = scores >= threshold
    positive = y_true == 1
    negative = ~positive
    tp = int(np.count_nonzero(predicted & positive))
    fp = int(np.count_nonzero(predicted & negative))
    tn = int(np.count_nonzero(~predicted & negative))
    fn = int(np.count_nonzero(~predicted & positive))
    precision = safe_ratio(tp, tp + fp)
    recall = safe_ratio(tp, tp + fn)
    specificity = safe_ratio(tn, tn + fp)
    npv = safe_ratio(tn, tn + fn)
    accuracy = safe_ratio(tp + tn, tp + fp + tn + fn)
    f1 = safe_ratio(2 * tp, 2 * tp + fp + fn)
    mcc_denominator = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = float((tp * tn - fp * fn) / mcc_denominator) if mcc_denominator else 0.0
    return {
        "model": model, "threshold_policy": threshold_policy, "threshold": float(threshold),
        "validation_rows": int(y_true.shape[0]), "positive_prevalence": float(y_true.mean()),
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "accuracy": accuracy, "balanced_accuracy": (recall + specificity) / 2,
        "precision": precision, "ppv": precision, "recall": recall,
        "sensitivity": recall, "specificity": specificity, "npv": npv,
        "f1": f1, "mcc": mcc,
        "alert_rate": safe_ratio(tp + fp, y_true.shape[0]),
        "alert_reduction": 1.0 - safe_ratio(tp + fp, y_true.shape[0]),
        **global_metrics,
    }


metric_rows = []
threshold_rows = []
for model_name, scores in predictions.items():
    if scores.shape != validation_truth.shape or not np.isfinite(scores).all():
        raise RuntimeError(f"Invalid validation predictions for {model_name}")
    global_metrics = {
        "auroc": float(roc_auc_score(validation_truth, scores)),
        "auprc": float(average_precision_score(validation_truth, scores)),
        "brier_score": float(brier_score_loss(validation_truth, scores)),
        "log_loss": float(log_loss(validation_truth, np.clip(scores, 1e-7, 1 - 1e-7))),
        "ece_15_bin": expected_calibration_error(validation_truth, scores),
    }
    selected_thresholds = {
        FIXED_THRESHOLD_POLICY: DECISION_THRESHOLD,
        "validation_max_f1_exploratory": max_f1_threshold(validation_truth, scores),
        "validation_recall_95_exploratory": recall_target_threshold(validation_truth, scores, TARGET_RECALL),
    }
    for policy, threshold in selected_thresholds.items():
        metric_rows.append(classification_metrics(
            model_name, validation_truth, scores, threshold, policy, global_metrics
        ))
    for threshold in np.unique(np.append(THRESHOLD_SWEEP, DECISION_THRESHOLD)):
        row = classification_metrics(
            model_name, validation_truth, scores, float(threshold), "threshold_sweep", global_metrics
        )
        threshold_rows.append(row)

metrics_by_threshold = pd.DataFrame(metric_rows)
threshold_sweep = pd.DataFrame(threshold_rows)
recorded_full_auc = float(gnn_marker["validation_auc"])
recomputed_full_auc = float(
    metrics_by_threshold.loc[metrics_by_threshold["model"] == "full_gnn", "auroc"].iloc[0]
)
if not math.isclose(recorded_full_auc, recomputed_full_auc, rel_tol=0, abs_tol=1e-6):
    raise RuntimeError(
        f"Full-GNN validation AUROC did not reproduce: {recomputed_full_auc} != {recorded_full_auc}"
    )
display(metrics_by_threshold.sort_values(["threshold_policy", "auroc"], ascending=[True, False]).round(6))
fixed_metrics = metrics_by_threshold[
    metrics_by_threshold["threshold_policy"] == FIXED_THRESHOLD_POLICY
].copy()
print("Fixed-threshold side-by-side comparison:")
display(fixed_metrics[[
    "model", "threshold", "auroc", "auprc", "accuracy", "precision",
    "recall", "f1", "specificity", "npv", "balanced_accuracy", "mcc",
]].sort_values("auroc", ascending=False).round(6))


In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve, roc_curve

PLOT_ROOT = DRIVE_STUDY / "plots"
PLOT_ROOT.mkdir(parents=True, exist_ok=True)
COLORS = {
    "full_gnn": "#006D77",
    "no_normalized_dosage": "#2A9D8F",
    "patient_only": "#E9C46A",
    "shuffled_topology": "#E76F51",
    "xgboost_saved_baseline": "#264653",
}
LABELS = {name: name.replace("_", " ").title() for name in predictions}
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 10})

figure, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for model_name, scores in predictions.items():
    fpr, tpr, _ = roc_curve(validation_truth, scores)
    precision, recall, _ = precision_recall_curve(validation_truth, scores)
    auc = roc_auc_score(validation_truth, scores)
    ap = average_precision_score(validation_truth, scores)
    axes[0].plot(fpr, tpr, color=COLORS[model_name], label=f"{LABELS[model_name]} ({auc:.3f})")
    axes[1].plot(recall, precision, color=COLORS[model_name], label=f"{LABELS[model_name]} ({ap:.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="#9AA0A6", linewidth=1)
axes[1].axhline(float(validation_truth.mean()), linestyle="--", color="#9AA0A6", linewidth=1)
axes[0].set(title="Validation ROC curves", xlabel="False-positive rate", ylabel="True-positive rate")
axes[1].set(title="Validation precision–recall curves", xlabel="Recall", ylabel="Precision")
for axis in axes:
    axis.grid(alpha=0.18)
    axis.legend(frameon=False, fontsize=8)
figure.tight_layout()
figure.savefig(PLOT_ROOT / "roc_pr_curves.png", dpi=180, bbox_inches="tight")
plt.show()

bar_metrics = ["accuracy", "precision", "recall", "f1"]
bar_frame = fixed_metrics.set_index("model").loc[list(predictions), bar_metrics]
axis = bar_frame.rename(index=LABELS).plot(
    kind="bar", figsize=(12, 5.5), color=["#457B9D", "#2A9D8F", "#E9C46A", "#E76F51"]
)
axis.set_title(f"Validation classification metrics at threshold {DECISION_THRESHOLD:.2f}")
axis.set_ylabel("Metric value")
axis.set_ylim(0, 1)
axis.grid(axis="y", alpha=0.18)
axis.legend(frameon=False, ncol=4)
axis.figure.tight_layout()
axis.figure.savefig(PLOT_ROOT / "fixed_threshold_metrics.png", dpi=180, bbox_inches="tight")
plt.show()

figure, axes = plt.subplots(2, 3, figsize=(12, 7.5))
for axis, model_name in zip(axes.flat, predictions, strict=False):
    row = fixed_metrics[fixed_metrics["model"] == model_name].iloc[0]
    matrix = np.array([[row["tn"], row["fp"]], [row["fn"], row["tp"]]], dtype=np.int64)
    axis.imshow(matrix, cmap="Blues")
    for row_index in range(2):
        for column_index in range(2):
            axis.text(column_index, row_index, f"{matrix[row_index, column_index]:,}", ha="center", va="center")
    axis.set_title(LABELS[model_name])
    axis.set_xticks((0, 1), labels=("Predicted 0", "Predicted 1"))
    axis.set_yticks((0, 1), labels=("Actual 0", "Actual 1"))
for axis in list(axes.flat)[len(predictions):]:
    axis.axis("off")
figure.suptitle(f"Validation confusion matrices at threshold {DECISION_THRESHOLD:.2f}")
figure.tight_layout()
figure.savefig(PLOT_ROOT / "confusion_matrices.png", dpi=180, bbox_inches="tight")
plt.show()

figure, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for model_name in predictions:
    local = threshold_sweep[threshold_sweep["model"] == model_name]
    axes[0].plot(local["threshold"], local["f1"], color=COLORS[model_name], label=LABELS[model_name])
    axes[1].plot(local["threshold"], local["recall"], color=COLORS[model_name], label=LABELS[model_name])
axes[0].axvline(DECISION_THRESHOLD, linestyle="--", color="#333333", linewidth=1)
axes[1].axvline(DECISION_THRESHOLD, linestyle="--", color="#333333", linewidth=1)
axes[1].axhline(TARGET_RECALL, linestyle=":", color="#C1121F", linewidth=1)
axes[0].set(title="F1 versus decision threshold", xlabel="Threshold", ylabel="F1")
axes[1].set(title="Recall versus decision threshold", xlabel="Threshold", ylabel="Recall")
for axis in axes:
    axis.set_ylim(0, 1)
    axis.grid(alpha=0.18)
    axis.legend(frameon=False, fontsize=8)
figure.tight_layout()
figure.savefig(PLOT_ROOT / "threshold_curves.png", dpi=180, bbox_inches="tight")
plt.show()

figure, axis = plt.subplots(figsize=(7, 6))
for model_name, scores in predictions.items():
    observed, predicted = calibration_curve(validation_truth, scores, n_bins=12, strategy="quantile")
    axis.plot(predicted, observed, marker="o", markersize=3, color=COLORS[model_name], label=LABELS[model_name])
axis.plot([0, 1], [0, 1], "--", color="#9AA0A6")
axis.set(title="Validation reliability curves", xlabel="Mean predicted probability", ylabel="Observed serious-outcome rate")
axis.grid(alpha=0.18)
axis.legend(frameon=False, fontsize=8)
figure.tight_layout()
figure.savefig(PLOT_ROOT / "calibration_curves.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
def write_parquet_atomic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_parquet(temporary, index=False, engine="pyarrow", compression="snappy")
    os.replace(temporary, path)


prediction_frame = pd.DataFrame({
    "primaryid": validation_primaryid,
    "split": "validation",
    "is_serious": validation_truth,
    **{f"score_{name}": scores for name, scores in predictions.items()},
})
prediction_path = DRIVE_STUDY / "validation_predictions.parquet"
metrics_path = DRIVE_STUDY / "metrics_by_threshold.parquet"
sweep_path = DRIVE_STUDY / "threshold_sweep.parquet"
write_parquet_atomic(prediction_frame, prediction_path)
write_parquet_atomic(metrics_by_threshold, metrics_path)
write_parquet_atomic(threshold_sweep, sweep_path)

relational_deltas = {}
model_auc = fixed_metrics.set_index("model")["auroc"].to_dict()
model_auprc = fixed_metrics.set_index("model")["auprc"].to_dict()
for comparator in ("patient_only", "shuffled_topology", "xgboost_saved_baseline"):
    relational_deltas[comparator] = {
        "delta_auroc_full_minus_comparator": float(model_auc["full_gnn"] - model_auc[comparator]),
        "delta_auprc_full_minus_comparator": float(model_auprc["full_gnn"] - model_auprc[comparator]),
    }

artifact_paths = [prediction_path, metrics_path, sweep_path, *sorted(PLOT_ROOT.glob("*.png"))]
study_manifest = {
    "format": "tekarx.gnn_ablation_study",
    "format_version": 1,
    "created_at_utc": datetime.now(UTC).isoformat(),
    "intended_use": "research decision-support evaluation; not diagnosis or causality",
    "study_id": STUDY_ID,
    "graph_checkpoint_id": graph_checkpoint_id,
    "gnn_checkpoint_id": gnn_checkpoint_id,
    "git_sha": RESOLVED_GIT_SHA,
    "versions": versions,
    "validation_period": "2024Q1",
    "test_period": "2024Q2",
    "test_evaluated": False,
    "validation_rows": int(validation_truth.shape[0]),
    "decision_threshold": DECISION_THRESHOLD,
    "target_recall": TARGET_RECALL,
    "threshold_policies": [
        FIXED_THRESHOLD_POLICY, "validation_max_f1_exploratory",
        "validation_recall_95_exploratory",
    ],
    "arms": list(predictions),
    "relational_deltas": relational_deltas,
    "artifacts": {
        path.relative_to(DRIVE_STUDY).as_posix(): {
            "size_bytes": path.stat().st_size, "sha256": sha256_file(path)
        }
        for path in artifact_paths
    },
    "limitations": [
        "Max-F1 and recall-target thresholds were selected and reported on the same validation period.",
        "The saved XGBoost baseline includes the graph's complete allow-listed patient matrix and is not perfectly feature-matched to the prospective GNN track.",
        "Split-local endpoint rotation can create duplicate patient-drug pairs.",
        "The one-hop GNN tests regimen-to-drug-feature aggregation, not multi-hop cross-patient propagation.",
        "Each neural arm uses one fixed random seed; repeat across seeds before making a strong scientific claim.",
    ],
}
manifest_path = DRIVE_STUDY / "ablation_manifest.json"
write_json_atomic(manifest_path, study_manifest)

summary_rows = metrics_by_threshold[
    metrics_by_threshold["threshold_policy"] == FIXED_THRESHOLD_POLICY
][["model", "auroc", "auprc", "accuracy", "precision", "recall", "f1", "specificity", "npv"]]
report_lines = [
    "# TekaRx GNN ablation study", "",
    f"Study ID: `{STUDY_ID}`", "",
    "Validation period: 2024Q1. Test period 2024Q2 remained locked.", "",
    f"Fixed decision threshold: {DECISION_THRESHOLD:.2f}. Target-recall policy: {TARGET_RECALL:.0%}.", "",
    "## Fixed-threshold metrics", "",
    summary_rows.to_markdown(index=False, floatfmt=".6f"), "",
    "## Relational deltas", "",
    "```json", json.dumps(relational_deltas, indent=2), "```", "",
    "Interpret positive full-minus-patient-only and full-minus-shuffled deltas as evidence that drug features and correct regimen assignment add validation signal. Small deltas indicate that most discrimination is already present in patient features.", "",
    "Thresholds optimized on validation are exploratory. Freeze the model and one threshold policy before a single 2024Q2 test evaluation.", "",
    "> Research decision support only; not a diagnosis or causal drug-safety conclusion.",
]
report_path = DRIVE_STUDY / "ablation_summary.md"
temporary_report = report_path.with_suffix(".md.tmp")
temporary_report.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
os.replace(temporary_report, report_path)
study_manifest["artifacts"]["ablation_summary.md"] = {
    "size_bytes": report_path.stat().st_size, "sha256": sha256_file(report_path)
}
write_json_atomic(manifest_path, study_manifest)
print(f"Durable study results: {DRIVE_STUDY}")
print(f"Manifest: {manifest_path}")
print("2024Q2 test labels were not evaluated.")


## How to interpret the result

1. Compare `full_gnn` against `patient_only`. A clear AUROC/AUPRC drop means the aggregated drug representation adds signal beyond patient features.
2. Compare `full_gnn` against `shuffled_topology`. A clear drop means the **correct patient–drug assignment** matters; a similar score means this graph formulation is getting little value from regimen alignment.
3. Compare `full_gnn` against `no_normalized_dosage` to quantify dosage's incremental contribution.
4. Compare with `xgboost_saved_baseline`, while remembering that the saved XGBoost feature set is not perfectly matched to the prospective GNN track.
5. Use AUROC for ranking, AUPRC for imbalanced performance, calibration/Brier score for probability quality, and precision/recall/F1 at a declared threshold for the intended alerting workflow. Accuracy alone can be misleading.

Do not unlock 2024Q2 after inspecting these results. First write down the chosen model, threshold policy, and all preprocessing decisions; then perform one final test evaluation.